<a href="https://colab.research.google.com/github/betulbilhan2/LLM-Augmentation-Fidelity/blob/main/NB05_Balanced_Subsampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05: Retention Dengeleme (Balanced Subsampling)
Bu notebook, bir önceki aşamada (`04_filtered`) kalite filtrelerinden geçip hayatta kalan Back-Translation (BT) ve LLM Paraphrase sentetik veri havuzlarını alır ve **her bir tohum (seed) bazında** sayılarını birbirine eşitler.

Amacımız, downstream model eğitimi (NB07) sırasında makalede yapılacak olan **Geleneksel (BT) vs. LLM veri artırımı** kıyaslamasının tamamen adil olmasını sağlamaktır. Yöntemlerden birinin filtreden daha fazla geçip daha çok örnekle temsil edilmesi, "bu yöntem daha iyidir" yanılgısına yol açmamalıdır. Model, her ikisinden de tam olarak eşit miktarda "temiz" veri gördüğünde hangisinden daha çok fayda sağlıyor, bunu ölçmek istiyoruz.

**Kesin Kurallarımız (V4 Protokolü):**
1. **Lokal Eşitleme (Seed-Based):** Eşitleme tüm deneyler için genel bir sayıya değil, *her bir tohumun (seed'in) kendi içindeki* minimum boyuta göre yapılır ($N_{min} = \min(N_{BT}, N_{LLM})$).
2. **Sınıf Dengeli (Stratified) Kesim ZORUNLULUĞU:** Büyük olan havuzdan örnekler atılırken, rastgele `sample` yapılmaz. Sınıf oranları (`label` veya `sentiment`) korunarak (stratified) alt örnekleme yapılır ki azınlık sınıfları (özellikle low-resource durumunda) şans eseri yok olmasın.
3. **Deterministik Rastgelelik (Reproducibility):** Rastgele kırpma işlemi tamamen tekrarlanabilir olmalıdır (`random_state=seed` kullanılarak).
4. **Sıfır Kalma Riski (Edge Case Control):** Düşük kaynak (low) senaryosunda bir yöntemin havuzu 0'a inerse kodun çökmemesi için $N_{min} = 0$ durumu kontrol altına alınmıştır.
5. **E0 ve E1'e Dokunmama Kuralı:** Orijinal veri (E0) ve kopya (E1) doğrudan eğitim sırasında okunacaktır, bu havuzlar filtrelenmediği için burada dengelenmeye tabi tutulmazlar.

In [ ]:
# 1. Kütüphanelerin Yüklenmesi ve Dizin Kurulumu
import os
import pandas as pd
import numpy as np
import yaml

# Drive Mount (Eğer Colab'da çalışıyorsa)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/tr_augmentation_project'
except:
    # Yerel ortam varsayımı
    BASE_DIR = 'C:/Users/btlbi/OneDrive/Masaüstü/TR Veri arttırımı'
    print("Colab ortamı bulunamadı, yerel dizin kullanılıyor:", BASE_DIR)

# Yeni çıktıların kaydedileceği 05_balanced/ ana klasörünün oluşturulması
os.makedirs(os.path.join(BASE_DIR, '05_balanced'), exist_ok=True)

for level in ['low', 'normal']:
    for method in ['backtranslation', 'llm_paraphrase']:
        os.makedirs(os.path.join(BASE_DIR, '05_balanced', level, method), exist_ok=True)

print("Dizinler hazırlandı.")

Mounted at /content/drive
Dizinler hazırlandı.


In [ ]:
# 2. Stratified Subsampling (Sınıf Dengeli Alt Örnekleme) Fonksiyonu
def stratified_subsample(df, target_size, label_col, random_state):
    """
    Verilen DataFrame'i, hedef boyuta sınıf oranlarını koruyarak (stratified) düşürür.
    Fazlalıkları rastgele ve deterministik olarak (random_state=seed) eler.
    """
    if len(df) <= target_size:
        return df.copy()
    if target_size == 0:
        return pd.DataFrame(columns=df.columns)

    counts = df[label_col].value_counts()
    proportions = counts / len(df)

    # Her sınıf için tam sayılara yuvarlanmış hedefleri hesapla
    target_counts = np.floor(proportions * target_size).astype(int)

    # Küsüratlardan dolayı eksik kalan kısımları (remainder) tamamla
    remainder = target_size - target_counts.sum()
    if remainder > 0:
        fractions = (proportions * target_size) - target_counts
        # Kesiri en büyük olan sınıflara öncelik ver.
        # Deterministik kırmak için class_name ile de sırala.
        df_frac = pd.DataFrame({'fraction': fractions, 'class_name': fractions.index})
        df_frac = df_frac.sort_values(by=['fraction', 'class_name'], ascending=[False, True])
        sorted_classes = df_frac['class_name'].tolist()

        for i in range(remainder):
            target_counts[sorted_classes[i]] += 1

    sampled_list = []
    for c, count in target_counts.items():
        if count > 0:
            class_df = df[df[label_col] == c]
            sampled_list.append(class_df.sample(n=count, random_state=random_state))

    # Listeyi birleştir ve sırasını karıştır (sınıf gruplaması oluşmasın)
    sampled_df = pd.concat(sampled_list)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return sampled_df

In [ ]:
# 3. Eşleştirme Döngüsü (Havuz Boyutlarının Keşfi ve Darboğazın Bulunması)

# Config'i yükle
config_path = os.path.join(BASE_DIR, 'configs', 'experiment_config.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    DATA_SEEDS = config['seeds']['data_seeds']
else:
    print("Config bulunamadı, varsayılan tohumlar (0-9) kullanılıyor.")
    DATA_SEEDS = list(range(10))

resource_levels = ['low', 'normal']
methods = ['backtranslation', 'llm_paraphrase']

balancing_records = []

for level in resource_levels:
    print(f"\n=== İşleniyor: Resource Level: {level.upper()} ===")
    for seed in DATA_SEEDS:
        bt_path = os.path.join(BASE_DIR, '04_filtered', level, 'backtranslation', f'seed_{seed}', 'pool_filtered.csv')
        llm_path = os.path.join(BASE_DIR, '04_filtered', level, 'llm_paraphrase', f'seed_{seed}', 'pool_filtered.csv')

        if not (os.path.exists(bt_path) and os.path.exists(llm_path)):
            print(f"  [Seed {seed}] 04_filtered altında veri bulunamadı, atlanıyor.")
            continue

        bt_df = pd.read_csv(bt_path)
        llm_df = pd.read_csv(llm_path)

        n_bt = len(bt_df)
        n_llm = len(llm_df)

        # İkisi arasındaki minimum değeri (darboğazı) bul
        n_min = min(n_bt, n_llm)

        label_col_bt = 'label' if 'label' in bt_df.columns else 'sentiment'
        label_col_llm = 'label' if 'label' in llm_df.columns else 'sentiment'

        print(f"  [Seed {seed}] BT: {n_bt}, LLM: {n_llm} -> Hedef Min Boyut: {n_min}")

        # Sıfır Kalma Riski (Edge Case Control)
        if n_min == 0:
            print(f"    Uyarı: seed {seed} için N_min = 0! Bypass yapılıyor, boş dosyalar kaydedilecek.")
            bt_balanced = pd.DataFrame(columns=bt_df.columns)
            llm_balanced = pd.DataFrame(columns=llm_df.columns)
        else:
            # Adil Alt Örnekleme (Hangi havuz büyükse ondan kırpılır)
            if n_bt > n_min:
                bt_balanced = stratified_subsample(bt_df, n_min, label_col_bt, random_state=seed)
                print(f"    BT havuzundan silindi: {n_bt} -> {n_min}")
            else:
                bt_balanced = bt_df.copy()

            if n_llm > n_min:
                llm_balanced = stratified_subsample(llm_df, n_min, label_col_llm, random_state=seed)
                print(f"    LLM havuzundan silindi: {n_llm} -> {n_min}")
            else:
                llm_balanced = llm_df.copy()

        # Kayıt ve Raporlama
        bt_out_dir = os.path.join(BASE_DIR, '05_balanced', level, 'backtranslation', f'seed_{seed}')
        llm_out_dir = os.path.join(BASE_DIR, '05_balanced', level, 'llm_paraphrase', f'seed_{seed}')

        os.makedirs(bt_out_dir, exist_ok=True)
        os.makedirs(llm_out_dir, exist_ok=True)

        bt_balanced.to_csv(os.path.join(bt_out_dir, 'pool_balanced.csv'), index=False)
        llm_balanced.to_csv(os.path.join(llm_out_dir, 'pool_balanced.csv'), index=False)

        balancing_records.append({
            'level': level,
            'seed': seed,
            'initial_BT_filtered': n_bt,
            'initial_LLM_filtered': n_llm,
            'final_balanced_size': n_min
        })

print("\nDengeleme (Balancing) işlemi başarıyla tamamlandı.")


=== İşleniyor: Resource Level: LOW ===
  [Seed 0] BT: 22, LLM: 28 -> Hedef Min Boyut: 22
    LLM havuzundan silindi: 28 -> 22
  [Seed 1] BT: 27, LLM: 29 -> Hedef Min Boyut: 27
    LLM havuzundan silindi: 29 -> 27
  [Seed 2] BT: 26, LLM: 29 -> Hedef Min Boyut: 26
    LLM havuzundan silindi: 29 -> 26
  [Seed 3] BT: 23, LLM: 26 -> Hedef Min Boyut: 23
    LLM havuzundan silindi: 26 -> 23
  [Seed 4] BT: 25, LLM: 28 -> Hedef Min Boyut: 25
    LLM havuzundan silindi: 28 -> 25
  [Seed 5] BT: 26, LLM: 30 -> Hedef Min Boyut: 26
    LLM havuzundan silindi: 30 -> 26
  [Seed 6] BT: 26, LLM: 27 -> Hedef Min Boyut: 26
    LLM havuzundan silindi: 27 -> 26
  [Seed 7] BT: 24, LLM: 26 -> Hedef Min Boyut: 24
    LLM havuzundan silindi: 26 -> 24
  [Seed 8] BT: 23, LLM: 27 -> Hedef Min Boyut: 23
    LLM havuzundan silindi: 27 -> 23
  [Seed 9] BT: 24, LLM: 28 -> Hedef Min Boyut: 24
    LLM havuzundan silindi: 28 -> 24

=== İşleniyor: Resource Level: NORMAL ===
  [Seed 0] BT: 242, LLM: 278 -> Hedef Min Boyut

In [ ]:
# 4. Raporlama: Dengeleme Sonuçlarının Gösterilmesi
if balancing_records:
    df_report = pd.DataFrame(balancing_records)
    print("Dengeleme (Balancing) Özeti:")
    import IPython.display as display
    display.display(df_report)

    # Özeti ayrıca bir CSV olarak 05_balanced klasörünün ana dizinine kaydet
    report_path = os.path.join(BASE_DIR, '05_balanced', 'balancing_summary.csv')
    df_report.to_csv(report_path, index=False)
    print(f"Özet tablo kaydedildi: {report_path}")

Dengeleme (Balancing) Özeti:


,level,seed,initial_BT_filtered,initial_LLM_filtered,final_balanced_size
0,low,0,22,28,22
1,low,1,27,29,27
2,low,2,26,29,26
3,low,3,23,26,23
4,low,4,25,28,25
5,low,5,26,30,26
6,low,6,26,27,26
7,low,7,24,26,24
8,low,8,23,27,23
9,low,9,24,28,24


Özet tablo kaydedildi: /content/drive/MyDrive/tr_augmentation_project/05_balanced/balancing_summary.csv
